# Pruebas clasificacion de olas

In [ ]:
import os
from pathlib import Path
import joblib
import numpy as np
from sklearn.metrics import silhouette_score as sil_score
from sklearn.cluster import DBSCAN
import itertools
from pyspark.sql import functions as F

In [ ]:
ambiente = 'dev'
PORCENTAJE_ENTRENAMIENTO = 0.5

## Seleccionar los datos

In [ ]:
data = (
    spark.sql(
        f"""
            SELECT coast_name, datetime, wind_u, wind_v, wave_u, wave_v, wave_period_s
            FROM cor_{ambiente}.silver.swell_metrics
        """
    )
)

In [ ]:
data_pre_processing = (
    data
    .withColumn('coast_year_month', F.concat(F.col('coast_name'), F.lit('_'), F.date_format('datetime', 'yyyy-MM')))
)

coast_year_month_dict = {row.coast_year_month: PORCENTAJE_ENTRENAMIENTO for row in data_pre_processing.select('coast_year_month').distinct().collect()}

data_sample = (
    data_pre_processing
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=0)
    .drop('coast_year_month')
).toPandas()

coast_names = data_sample['coast_name'].unique()

## Preparar los datos

In [ ]:
features = ['wind_u', 'wind_v', 'wave_u', 'wave_v', 'wave_period_s']

In [ ]:
if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    scaler_path = f'/Volumes/cor_{ambiente}/ml/models/scaler/scaler_{{}}.pkl'
else:
    base_path = Path.cwd().parent
    scaler_path = f'{base_path}/scaler/scaler_{{}}.pkl'

In [ ]:
X = np.empty((0, len(features)))
for coast_name in coast_names:
    scaler_path_coast = scaler_path.format(coast_name)
    scaler = joblib.load(scaler_path_coast)
    X_coast = data_sample[data_sample['coast_name'] == coast_name][features]
    X_coast_scaled = scaler.transform(X_coast)
    X = np.vstack((X, X_coast_scaled))

# Preparar pruebas de parametros

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
posible_e = np.linspace(0.01, 1, 15)
sample_5 = X.shape[0] // 5
posible_min_samples = np.arange(int(sample_5 * 0.7), int(sample_5 * 1.2), 3)
combinaciones = list(itertools.product(posible_e, posible_min_samples))
print(len(combinaciones))

In [ ]:
# https://www.youtube.com/watch?v=VO_uzCU_nKw
def get_scores_and_labels(combinations, X):
  parametros_ok = []

  for i, (eps, num_samples) in enumerate(combinations):
    dbscan_cluster_model = DBSCAN(eps=eps, min_samples=num_samples).fit(X)
    labels = dbscan_cluster_model.labels_
    labels_set = set(labels)
    num_clusters = len(labels_set)
    if -1 in labels_set:
      num_clusters -= 1
    
    if (num_clusters < 2) or (num_clusters > 5):
      continue
    
    parametros_ok.append({
        'eps': eps,
        'num_samples': num_samples,
        'labels': labels,
        'score': sil_score(X, labels)
    })

  return parametros_ok

parametros_ok = get_scores_and_labels(combinaciones, X)

In [ ]:
px.scatter(
    x=[param['eps'] for param in parametros_ok],
    y=[param['num_samples'] for param in parametros_ok],
    color=[param['score'] for param in parametros_ok],
    color_continuous_scale='Viridis',
    title='DBSCAN Silhouette Scores'
)

In [ ]:
best_index = np.argmax([param['score'] for param in parametros_ok])

In [ ]:
data_sample['cluster'] = parametros_ok[best_index]['labels']

In [ ]:
data_sample['cluster'].value_counts()

In [ ]:
# generar uns imagen con 2 columnas y 1 fila, cada una con un gráfico de dispersión wave_u vs wave_v, pero con un color diferente para cada cluster
fig = make_subplots(rows=2, cols=len(coast_names), subplot_titles=coast_names)
for i, coast_name in enumerate(coast_names):
    data_coast = data_sample[data_sample['coast_name'] == coast_name]
    fig.add_trace(
        go.Scatter(
            x=data_coast['wave_u'],
            y=data_coast['wave_v'],
            mode='markers',
            marker=dict(color=data_coast['cluster'], colorscale='Viridis', showscale=False)
        ),
        row=1, col=i+1
    )
    fig.add_trace(
        go.Scatter(
            x=data_coast['wind_u'],
            y=data_coast['wind_v'],
            mode='markers',
            marker=dict(color=data_coast['cluster'], colorscale='Viridis', showscale=False)
        ),
        row=2, col=i+1
    )

fig.show()